# Analisis Georreferenciado con MAPBIOMAS.

In [1]:
from scripts.config import DATA_DIR
import polars as pl
import os


raw_folder = DATA_DIR / "raw" / "MAPBIOMAS"
f0 = os.listdir(raw_folder)[0]
file = pl.read_parquet(raw_folder / f0)
file

lon,lat,clase_id,id_name
f64,f64,u8,str
-114.72398,32.718394,19,"""Cultivo Anual"""
-114.72371,32.718394,19,"""Cultivo Anual"""
-114.723441,32.718394,19,"""Cultivo Anual"""
-114.723171,32.718394,19,"""Cultivo Anual"""
-114.722902,32.718394,19,"""Cultivo Anual"""
…,…,…,…
-92.228728,14.534517,19,"""Cultivo Anual"""
-92.228458,14.534517,19,"""Cultivo Anual"""
-92.228189,14.534517,19,"""Cultivo Anual"""


In [2]:
import re

with open("agrocoords.txt", "r") as txt:
    agropoints = txt.read()
    agropoints = agropoints.split("\n")
    agropoints = agropoints[:-1]
agropoints

['(-102.28591602335558, 21.72969343327807)',
 '(-102.40975006811368, 21.77243531020748)',
 '(-102.51579591803714, 21.795837141742023)',
 '(-102.4688593743015, 21.810878688139404)',
 '(-102.54260394243583, 21.815668400078007)',
 '(-102.47401208108364, 21.82111352485242)',
 '(-102.50706015594378, 21.824792441987075)',
 '(-102.5353547274295, 21.850170059294136)',
 '(-102.30092512226852, 21.744920018325043)',
 '(-102.36349321269195, 21.750914530967385)',
 '(-102.3405643280743, 21.766569080950305)',
 '(-102.32019671495242, 21.77945372415422)',
 '(-102.30410332172106, 21.7850455482622)',
 '(-102.30532013432367, 21.79488474179825)',
 '(-102.3366257919657, 21.799175103435783)',
 '(-102.26830043879052, 21.802961308395794)',
 '(-102.27065937253349, 21.814009134288003)',
 '(-102.27052003010203, 21.837922361046775)',
 '(-102.78681711569446, 21.823016866206785)',
 '(-102.12012915878556, 21.83274016112649)',
 '(-102.11251584526886, 21.868210991013513)',
 '(-102.07252865853124, 21.93334212423573)',
 

In [3]:
serie_coords = []

for point in agropoints:
    b = point.split(",")
    c = [b1.strip() for b1 in b]
    d = [float(c[0][1:]), float(c[1][:-1])]
    serie_coords.append(d)

serie_coords

[[-102.28591602335558, 21.72969343327807],
 [-102.40975006811368, 21.77243531020748],
 [-102.51579591803714, 21.795837141742023],
 [-102.4688593743015, 21.810878688139404],
 [-102.54260394243583, 21.815668400078007],
 [-102.47401208108364, 21.82111352485242],
 [-102.50706015594378, 21.824792441987075],
 [-102.5353547274295, 21.850170059294136],
 [-102.30092512226852, 21.744920018325043],
 [-102.36349321269195, 21.750914530967385],
 [-102.3405643280743, 21.766569080950305],
 [-102.32019671495242, 21.77945372415422],
 [-102.30410332172106, 21.7850455482622],
 [-102.30532013432367, 21.79488474179825],
 [-102.3366257919657, 21.799175103435783],
 [-102.26830043879052, 21.802961308395794],
 [-102.27065937253349, 21.814009134288003],
 [-102.27052003010203, 21.837922361046775],
 [-102.78681711569446, 21.823016866206785],
 [-102.12012915878556, 21.83274016112649],
 [-102.11251584526886, 21.868210991013513],
 [-102.07252865853124, 21.93334212423573],
 [-102.4542460534553, 22.162783053927793],
 [

In [4]:
import numpy as np

serie = pl.DataFrame(serie_coords, orient='row')
serie.columns = ['lon', 'lat']
serie = serie.with_columns(pl.col("lon").truncate(0).alias('lonE'))
lon_blocks = serie['lonE'].unique().to_list()

In [65]:
print(lon_blocks[0])
serie_lon = serie.filter(np.abs(pl.col("lon") - lon_blocks[0]) < 1)
serie_lon

-117.0


lon,lat,lonE
f64,f64,f64
-116.019221,30.499964,-116.0
-116.034311,30.754053,-116.0
-116.051156,30.789154,-116.0
-116.029607,30.808839,-116.0
-116.004176,30.811962,-116.0
…,…,…
-116.736332,32.551201,-116.0
-116.571029,32.564532,-116.0
-116.754582,32.55143,-116.0


In [66]:
agro_lon = file.filter(np.abs(pl.col("lon") - lon_blocks[0]) < 1)


In [76]:
agro_lon

lon,lat,clase_id,id_name
f64,f64,u8,str
-116.139635,32.615717,19,"""Cultivo Anual"""
-116.139365,32.615717,19,"""Cultivo Anual"""
-116.138287,32.615717,19,"""Cultivo Anual"""
-116.138018,32.615717,19,"""Cultivo Anual"""
-116.137748,32.615717,19,"""Cultivo Anual"""
…,…,…,…
-116.000306,30.361934,19,"""Cultivo Anual"""
-116.000037,30.361934,19,"""Cultivo Anual"""
-116.000306,30.361664,19,"""Cultivo Anual"""


In [ ]:
import pandas as pd


v = pd.DataFrame(index = range(len(serie_lon)), columns = range(len(agro_lon)))



TypeError: cannot unpack non-iterable int object

In [77]:
for i in range(v.shape[0]):
    print(i)
    for j in range(v.shape[1]):
        p1lon = serie_lon[i]["lon"]
        p1lat = serie_lon[i]["lat"]
        p2lon = agro_lon[j]['lon']
        p2lat = agro_lon[j]['lat']
        v.loc[i, j] = np.sqrt((p2lon - p1lon) ** 2 + (p2lat - p1lat) ** 2)


0


KeyboardInterrupt: 

In [78]:
serie

lon,lat,lonE
f64,f64,f64
-102.285916,21.729693,-102.0
-102.40975,21.772435,-102.0
-102.515796,21.795837,-102.0
-102.468859,21.810879,-102.0
-102.542604,21.815668,-102.0
…,…,…
-101.704667,24.756222,-101.0
-101.7887,24.863937,-101.0
-102.124152,24.783342,-102.0


In [80]:
for i in range(len(serie)):
    print(serie[i])

shape: (1, 3)
┌─────────────┬───────────┬────────┐
│ lon         ┆ lat       ┆ lonE   │
│ ---         ┆ ---       ┆ ---    │
│ f64         ┆ f64       ┆ f64    │
╞═════════════╪═══════════╪════════╡
│ -102.285916 ┆ 21.729693 ┆ -102.0 │
└─────────────┴───────────┴────────┘
shape: (1, 3)
┌────────────┬───────────┬────────┐
│ lon        ┆ lat       ┆ lonE   │
│ ---        ┆ ---       ┆ ---    │
│ f64        ┆ f64       ┆ f64    │
╞════════════╪═══════════╪════════╡
│ -102.40975 ┆ 21.772435 ┆ -102.0 │
└────────────┴───────────┴────────┘
shape: (1, 3)
┌─────────────┬───────────┬────────┐
│ lon         ┆ lat       ┆ lonE   │
│ ---         ┆ ---       ┆ ---    │
│ f64         ┆ f64       ┆ f64    │
╞═════════════╪═══════════╪════════╡
│ -102.515796 ┆ 21.795837 ┆ -102.0 │
└─────────────┴───────────┴────────┘
shape: (1, 3)
┌─────────────┬───────────┬────────┐
│ lon         ┆ lat       ┆ lonE   │
│ ---         ┆ ---       ┆ ---    │
│ f64         ┆ f64       ┆ f64    │
╞═════════════╪═══════════